In [2]:
import pyautogui
import pandas as pd
import numpy as np
from importlib import reload
from source import select


## Read in telemetry file

### **select file**

In [18]:
reload(select)

ifile = 
#ifile = select.single_file()

ifile

### **read data into dictionary**

In [11]:
import re
from collections import defaultdict

data = defaultdict(lambda: defaultdict(list))

with open(ifile, 'r') as f:
    for line in f:
        try:
            match = re.match(r'\$(\w+),(.*)\*(\w+)', line.strip())
            if not match:
                continue
            tm_type, content = match.group(1), match.group(2)
            fields = {}
            for field in content.split(','):
                if '=' in field:
                    k, v = field.split('=', 1)
                    fields[k] = v
            for key, value in fields.items():
                data[tm_type][key].append(value)
        except Exception as e:
            print(f"Skipping line due to error: {line.strip()}\n{e}")

data.keys()

dict_keys(['PNORI2', 'PNORS2', 'PNORC2'])

### **check contents**

In [12]:
pd.DataFrame(data['PNORC2'])

,DATE,TIME,CN,CP,VE,VN,VU,VU2,A1,A2,A3,A4,C1,C2,C3,C4
0,101226,112113,1,2.0,-0.055,-0.024,-0.022,-0.021,76.7,74.5,75.0,75.0,86,94,94,96
1,101226,112113,2,3.0,-0.028,0.016,-0.017,-0.016,70.6,69.2,69.9,70.2,99,97,99,95
2,101226,112113,3,4.0,-0.048,0.007,0.000,-0.012,67.2,67.2,67.6,67.9,95,100,98,100
3,101226,112113,4,5.0,-0.035,-0.051,0.007,0.007,64.3,64.3,63.8,64.8,100,100,100,99
4,101226,112113,5,6.0,-0.042,-0.097,0.022,-0.011,62.8,61.8,60.1,62.9,100,97,94,99
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4523,101226,112623,29,30.0,-0.590,0.507,-0.012,-0.150,35.9,38.8,40.5,35.5,81,88,86,71
4524,101226,112625,1,2.0,-0.011,0.018,0.007,0.001,76.1,75.5,75.8,74.9,98,98,97,90
4525,101226,112625,2,3.0,-0.091,-0.019,-0.019,0.019,69.5,70.2,70.3,71.0,94,93,96,99
4526,101226,112625,3,4.0,-0.159,-0.003,-0.001,0.025,66.6,67.0,66.4,67.0,99,98,97,97


## Save and format data

### **convert to xarray.Dataset**

In [13]:
import xarray as xr
import pandas as pd

def parse_time(df):
    df['TIME'] = pd.to_datetime(df['DATE']+' '+df['TIME'], format='%m%d%y %H%M%S')
    return df.drop(columns='DATE')

try:
    # Process PNORC2
    cdf = parse_time(pd.DataFrame(data['PNORC2']))
    cdf['CN'] = cdf['CN'].astype(int)
    cdf = cdf.set_index(['TIME', 'CN'])
    ds = cdf.to_xarray()
    
    # Process PNORS2
    sdf = parse_time(pd.DataFrame(data['PNORS2'])).set_index('TIME')
    for col in sdf.columns:
        ds[col] = ('TIME', sdf[col])
    
    # Add PNORI2
    for k, v in data['PNORI2'].items():
        ds[k] = v[0]

except KeyError as e:
    print(f"Missing expected keys in data: {e}")
    ds = xr.Dataset()

ds

<xarray.Dataset> Size: 491kB
Dimensions:  (TIME: 157, CN: 29)
Coordinates:
  * TIME     (TIME) datetime64[ns] 1kB 2026-10-12T11:21:13 ... 2026-10-12T11:...
  * CN       (CN) int64 232B 1 2 3 4 5 6 7 8 9 10 ... 21 22 23 24 25 26 27 28 29
Data variables: (12/33)
    CP       (TIME, CN) object 36kB '2.0' '3.0' '4.0' '5.0' ... nan nan nan nan
    VE       (TIME, CN) object 36kB '-0.055' '-0.028' '-0.048' ... nan nan nan
    VN       (TIME, CN) object 36kB '-0.024' '0.016' '0.007' ... nan nan nan
    VU       (TIME, CN) object 36kB '-0.022' '-0.017' '0.000' ... nan nan nan
    VU2      (TIME, CN) object 36kB '-0.021' '-0.016' '-0.012' ... nan nan nan
    A1       (TIME, CN) object 36kB '76.7' '70.6' '67.2' '64.3' ... nan nan nan
    ...       ...
    SN       <U6 24B '106516'
    NB       <U1 4B '4'
    NC       <U2 8B '29'
    BD       <U4 16B '1.00'
    CS       <U4 16B '1.00'
    CY       <U3 12B 'ENU'

In [14]:
import numpy as np

def assign_dtypes(ds, dtypes, fill=None):
    for col in ds.variables:
        if col in dtypes.keys():
            ds[col] = ds[col].astype(dtypes[col])
        else:
            if fill: 
                ds[col] = ds[col].astype(fill)

dtypes = dict(
    TIME = np.datetime64,
    CN = int,
    EC = int,
    IT = int,
    NB = int,
    NC = int,
    SC = str,
    SN = str,
    CY = str
)

assign_dtypes(ds, dtypes, fill=float)

ds

<xarray.Dataset> Size: 495kB
Dimensions:  (TIME: 157, CN: 29)
Coordinates:
  * TIME     (TIME) datetime64[ns] 1kB 2026-10-12T11:21:13 ... 2026-10-12T11:...
  * CN       (CN) int64 232B 1 2 3 4 5 6 7 8 9 10 ... 21 22 23 24 25 26 27 28 29
Data variables: (12/33)
    CP       (TIME, CN) float64 36kB 2.0 3.0 4.0 5.0 6.0 ... nan nan nan nan nan
    VE       (TIME, CN) float64 36kB -0.055 -0.028 -0.048 -0.035 ... nan nan nan
    VN       (TIME, CN) float64 36kB -0.024 0.016 0.007 -0.051 ... nan nan nan
    VU       (TIME, CN) float64 36kB -0.022 -0.017 0.0 0.007 ... nan nan nan nan
    VU2      (TIME, CN) float64 36kB -0.021 -0.016 -0.012 0.007 ... nan nan nan
    A1       (TIME, CN) float64 36kB 76.7 70.6 67.2 64.3 ... nan nan nan nan
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 29
    BD       float64 8B 1.0
    CS       float64 8B 1.0
    CY       <U3 12B 'ENU'

In [15]:
print(ds)

<xarray.Dataset> Size: 495kB
Dimensions:  (TIME: 157, CN: 29)
Coordinates:
  * TIME     (TIME) datetime64[ns] 1kB 2026-10-12T11:21:13 ... 2026-10-12T11:...
  * CN       (CN) int64 232B 1 2 3 4 5 6 7 8 9 10 ... 21 22 23 24 25 26 27 28 29
Data variables: (12/33)
    CP       (TIME, CN) float64 36kB 2.0 3.0 4.0 5.0 6.0 ... nan nan nan nan nan
    VE       (TIME, CN) float64 36kB -0.055 -0.028 -0.048 -0.035 ... nan nan nan
    VN       (TIME, CN) float64 36kB -0.024 0.016 0.007 -0.051 ... nan nan nan
    VU       (TIME, CN) float64 36kB -0.022 -0.017 0.0 0.007 ... nan nan nan nan
    VU2      (TIME, CN) float64 36kB -0.021 -0.016 -0.012 0.007 ... nan nan nan
    A1       (TIME, CN) float64 36kB 76.7 70.6 67.2 64.3 ... nan nan nan nan
    ...       ...
    SN       <U6 24B '106516'
    NB       int64 8B 4
    NC       int64 8B 29
    BD       float64 8B 1.0
    CS       float64 8B 1.0
    CY       <U3 12B 'ENU'


### **export to netCDF**

In [17]:
import os

filename = os.path.basename(ifile).rpartition('.')[0]
ofile = f'data/netcdf/{filename}.nc'

ds.to_netcdf(ofile)